# 🛡️ Automata-Based Spam Filter / Keyword Detector
### Using FSA + Regular Expressions to Detect Spam Patterns
**Focus:** Keyword Matching | Pattern Complexity | False Positive Reduction

---
## 📦 Imports & Setup

In [ ]:
import re
from dataclasses import dataclass, field
from typing import List, Dict, Tuple, Optional
from collections import defaultdict
import textwrap

print("✅ Libraries loaded successfully.")

---
## 🔤 Part 1 — Keyword Dictionary (Weighted Spam Lexicon)

Keywords are categorized by **domain** and assigned **weights** (1–3) based on spam likelihood.
Higher weight = stronger spam signal.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Weighted Spam Keyword Dictionary
# Format: { category: [(keyword, weight), ...] }
# ─────────────────────────────────────────────────────────────
SPAM_KEYWORDS: Dict[str, List[Tuple[str, int]]] = {
    "financial": [
        ("free money", 3),
        ("cash prize", 3),
        ("earn \\$", 2),
        ("make money fast", 3),
        ("double your income", 3),
        ("financial freedom", 2),
        ("risk free", 2),
        ("guaranteed profit", 3),
        ("no investment", 2),
        ("wire transfer", 2),
    ],
    "urgency": [
        ("act now", 2),
        ("limited time offer", 2),
        ("expires today", 2),
        ("urgent", 1),
        ("don't miss out", 2),
        ("last chance", 2),
        ("respond immediately", 2),
        ("hurry", 1),
    ],
    "lottery_prize": [
        ("you have won", 3),
        ("congratulations you", 2),
        ("claim your prize", 3),
        ("selected winner", 3),
        ("lottery winner", 3),
        ("lucky draw", 2),
        ("prize notification", 2),
    ],
    "phishing": [
        ("verify your account", 3),
        ("confirm your password", 3),
        ("click here to verify", 3),
        ("update your billing", 3),
        ("your account has been suspended", 3),
        ("login details", 2),
        ("bank account number", 3),
        ("social security", 2),
    ],
    "health": [
        ("lose weight fast", 3),
        ("miracle cure", 3),
        ("doctor approved", 2),
        ("buy cheap meds", 3),
        ("no prescription", 2),
        ("100% natural", 1),
        ("burn fat", 2),
    ],
    "adult": [
        ("meet singles", 2),
        ("hot girls", 3),
        ("click for adult", 3),
        ("webcam girls", 3),
    ],
}

total_keywords = sum(len(v) for v in SPAM_KEYWORDS.values())
print(f"📚 Spam keyword dictionary loaded: {total_keywords} keywords across {len(SPAM_KEYWORDS)} categories.")
for cat, kws in SPAM_KEYWORDS.items():
    print(f"   [{cat}] → {len(kws)} keywords")

---
## ⚙️ Part 2 — Regex Pattern Engine

Compiles weighted keywords into regex patterns with:
- **Case-insensitive** matching
- **Word-boundary** anchors to reduce false positives
- **Leet-speak** normalization (e.g., `fr3e` → `free`)
- **Punctuation/space variation** handling (`f.r.e.e`, `f r e e`)

In [ ]:
@dataclass
class PatternMatch:
    keyword: str
    category: str
    weight: int
    matched_text: str
    position: Tuple[int, int]


class RegexPatternEngine:
    """
    Compiles spam keywords into optimized regex patterns.
    Handles leet-speak, spacing variations, and punctuation noise.
    """

    # Leet-speak normalization map
    LEET_MAP = {
        '0': 'o', '1': 'i', '3': 'e', '4': 'a',
        '5': 's', '7': 't', '@': 'a', '$': 's',
    }

    def __init__(self, keyword_dict: Dict[str, List[Tuple[str, int]]]):
        self.keyword_dict = keyword_dict
        self.compiled_patterns: List[Tuple[re.Pattern, str, int, str]] = []
        self._compile_all()

    def _normalize_leet(self, text: str) -> str:
        """Replace leet characters with standard letters."""
        for leet, normal in self.LEET_MAP.items():
            text = text.replace(leet, normal)
        return text

    def _keyword_to_regex(self, keyword: str) -> str:
        """
        Convert a keyword to a flexible regex pattern:
        - Allows optional punctuation/spaces between characters within words
        - Anchors on word boundaries for multi-word phrases
        """
        words = keyword.strip().split()
        word_patterns = []
        for word in words:
            escaped = re.escape(word)
            # Allow optional dot/dash/space between letters (e.g., f.r.e.e)
            flexible = r'[.\-\s]?'.join(list(escaped))
            word_patterns.append(flexible)
        # Join words with flexible whitespace
        pattern = r'\s+'.join(word_patterns)
        return r'\b' + pattern + r'\b'

    def _compile_all(self):
        """Compile all keywords into regex patterns."""
        for category, keywords in self.keyword_dict.items():
            for keyword, weight in keywords:
                pattern_str = self._keyword_to_regex(keyword)
                try:
                    compiled = re.compile(pattern_str, re.IGNORECASE | re.UNICODE)
                    self.compiled_patterns.append((compiled, category, weight, keyword))
                except re.error as e:
                    print(f"⚠️  Regex compile error for '{keyword}': {e}")

    def match(self, text: str) -> List[PatternMatch]:
        """Find all keyword matches in text after leet normalization."""
        normalized = self._normalize_leet(text)
        matches = []
        for compiled, category, weight, keyword in self.compiled_patterns:
            for m in compiled.finditer(normalized):
                matches.append(PatternMatch(
                    keyword=keyword,
                    category=category,
                    weight=weight,
                    matched_text=m.group(),
                    position=(m.start(), m.end())
                ))
        return matches


engine = RegexPatternEngine(SPAM_KEYWORDS)
print(f"✅ Regex Pattern Engine compiled: {len(engine.compiled_patterns)} patterns ready.")

---
## 🤖 Part 3 — Finite State Automaton (FSA) for Spam Detection

The FSA processes each email through a sequence of states based on accumulated spam signals:

```
  [START] ──low signal──► [CLEAN]
     │
     ├──medium signal──► [SUSPICIOUS]
     │
     └──high signal───► [SPAM]
```

Transitions are driven by **weighted score** from regex matches.

In [ ]:
# ─────────────────────────────────────────────────────────────
# FSA State Definitions
# ─────────────────────────────────────────────────────────────
FSA_STATES = ['START', 'CLEAN', 'SUSPICIOUS', 'SPAM']
ACCEPT_STATES = ['SPAM']
REJECT_STATES = ['CLEAN']

# Score thresholds for state transitions
THRESHOLD_SUSPICIOUS = 3   # score >= 3 → SUSPICIOUS
THRESHOLD_SPAM       = 7   # score >= 7 → SPAM

@dataclass
class FSAResult:
    final_state: str
    score: int
    verdict: str
    matches: List[PatternMatch]
    category_breakdown: Dict[str, int]
    false_positive_flags: List[str]
    state_trace: List[str]


class SpamFSA:
    """
    Finite State Automaton for spam classification.
    
    States: START → CLEAN | SUSPICIOUS | SPAM
    Input:  Weighted regex match score + contextual signals
    Output: Classification verdict with confidence
    """

    # False positive reduction: whitelist contexts
    # If these patterns exist, reduce suspicion for matched keywords
    FP_WHITELIST_PATTERNS = [
        (r'\bunsubscribe\b', -1),            # Legitimate marketing footer
        (r'\bprivacy policy\b', -1),         # Legitimate footer
        (r'\bterms (of|and) (service|use)\b', -1),
        (r'\b(dear|hello|hi)\s+[A-Z][a-z]+\b', -1),  # Personalized greeting
        (r'\bthis email was sent to\b', -1), # Legit newsletter
        (r'\bsafe and secure\b', -1),
        (r'\bno spam\b', -1),
    ]

    # Spam amplifiers: if these exist, boost the score
    AMPLIFIER_PATTERNS = [
        (r'!!!+', 2),                        # Multiple exclamation marks
        (r'\$\d+', 1),                       # Dollar amounts
        (r'[A-Z]{5,}', 1),                   # SHOUTING in caps
        (r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', 1),  # Phone number
        (r'https?://\S+\.tk|https?://bit\.ly', 2),  # Suspicious URLs
        (r'click\s+here', 2),
        (r'\bfree\b', 1),
        (r'\b(winner|won|prize)\b', 1),
    ]

    def __init__(self, pattern_engine: RegexPatternEngine):
        self.engine = pattern_engine
        # FSA transition table: (state, signal) → next_state
        self.transitions = {
            ('START',      'low'):    'CLEAN',
            ('START',      'medium'): 'SUSPICIOUS',
            ('START',      'high'):   'SPAM',
            ('CLEAN',      'low'):    'CLEAN',
            ('CLEAN',      'medium'): 'SUSPICIOUS',
            ('CLEAN',      'high'):   'SPAM',
            ('SUSPICIOUS', 'low'):    'SUSPICIOUS',
            ('SUSPICIOUS', 'medium'): 'SUSPICIOUS',
            ('SUSPICIOUS', 'high'):   'SPAM',
            ('SPAM',       'low'):    'SPAM',
            ('SPAM',       'medium'): 'SPAM',
            ('SPAM',       'high'):   'SPAM',
        }

    def _score_to_signal(self, score: int) -> str:
        if score < THRESHOLD_SUSPICIOUS:
            return 'low'
        elif score < THRESHOLD_SPAM:
            return 'medium'
        return 'high'

    def _compute_fp_adjustment(self, text: str) -> Tuple[int, List[str]]:
        """Check whitelist/amplifier patterns and return score adjustment."""
        adjustment = 0
        flags = []
        text_lower = text.lower()

        for pattern, delta in self.FP_WHITELIST_PATTERNS:
            if re.search(pattern, text_lower, re.IGNORECASE):
                adjustment += delta
                flags.append(f"✅ FP-Reducer: '{pattern}' (score {delta:+d})")

        for pattern, delta in self.AMPLIFIER_PATTERNS:
            if re.search(pattern, text, re.IGNORECASE):
                adjustment += delta
                flags.append(f"🔴 Amplifier: '{pattern}' (score {delta:+d})")

        return adjustment, flags

    def classify(self, text: str) -> FSAResult:
        """Run the FSA on input text and return a classification result."""
        state = 'START'
        state_trace = ['START']

        # Step 1: Regex keyword matching
        matches = self.engine.match(text)
        base_score = sum(m.weight for m in matches)

        # Step 2: Category breakdown
        cat_breakdown: Dict[str, int] = defaultdict(int)
        for m in matches:
            cat_breakdown[m.category] += m.weight

        # Step 3: False positive adjustment & amplifiers
        fp_adjustment, fp_flags = self._compute_fp_adjustment(text)
        total_score = max(0, base_score + fp_adjustment)

        # Step 4: FSA transition
        signal = self._score_to_signal(total_score)
        next_state = self.transitions.get((state, signal), 'CLEAN')
        state_trace.append(next_state)
        state = next_state

        # Verdict
        verdict_map = {
            'CLEAN':      '✅ HAM (Not Spam)',
            'SUSPICIOUS': '⚠️  SUSPICIOUS',
            'SPAM':       '🚫 SPAM',
        }

        return FSAResult(
            final_state=state,
            score=total_score,
            verdict=verdict_map.get(state, 'UNKNOWN'),
            matches=matches,
            category_breakdown=dict(cat_breakdown),
            false_positive_flags=fp_flags,
            state_trace=state_trace,
        )


fsa = SpamFSA(engine)
print("✅ SpamFSA initialized with", len(fsa.transitions), "state transitions.")

---
## 📋 Part 4 — Pretty Report Printer

In [ ]:
def print_report(email_text: str, result: FSAResult, label: Optional[str] = None):
    """Print a formatted analysis report for a single email."""
    width = 68
    print("═" * width)
    if label:
        print(f" 📧 {label}")
    print(f" TEXT : {textwrap.shorten(email_text, width=55)}")
    print("─" * width)

    # FSA Trace
    trace_str = " → ".join(result.state_trace)
    print(f" FSA TRACE    : {trace_str}")
    print(f" FINAL STATE  : {result.final_state}")
    print(f" SPAM SCORE   : {result.score}  "
          f"(suspicious≥{THRESHOLD_SUSPICIOUS}, spam≥{THRESHOLD_SPAM})")
    print(f" VERDICT      : {result.verdict}")
    print("─" * width)

    # Keyword matches
    if result.matches:
        print(f" KEYWORD MATCHES ({len(result.matches)}):")
        for m in result.matches:
            print(f"   [{m.category}] '{m.matched_text}' "
                  f"(keyword: '{m.keyword}', weight: {m.weight}, pos: {m.position})")
    else:
        print(" KEYWORD MATCHES: None")

    # Category breakdown
    if result.category_breakdown:
        print("─" * width)
        print(" CATEGORY SCORE BREAKDOWN:")
        for cat, score in sorted(result.category_breakdown.items(),
                                  key=lambda x: -x[1]):
            bar = '█' * score
            print(f"   {cat:<20} {bar} ({score})")

    # False positive / amplifier signals
    if result.false_positive_flags:
        print("─" * width)
        print(" CONTEXTUAL SIGNALS:")
        for flag in result.false_positive_flags:
            print(f"   {flag}")

    print("═" * width)
    print()

print("✅ Report printer ready.")

---
## 🧪 Part 5 — Test Emails Dataset

In [ ]:
test_emails = [
    (
        "Classic Lottery Spam",
        """CONGRATULATIONS!!! You have WON a CASH PRIZE of $50,000!
        You are our selected winner in the lucky draw. ACT NOW!
        Claim your prize by clicking here: http://bit.ly/win-now
        Don't miss out on this LIMITED TIME OFFER. Respond immediately!"""
    ),
    (
        "Phishing Attack",
        """Dear Customer, your account has been suspended.
        Please verify your account and confirm your password immediately.
        Click here to verify: http://secure-login.tk/update
        You must update your billing details within 24 hours to avoid closure."""
    ),
    (
        "Health Spam",
        """Lose weight FAST with our MIRACLE CURE!!!
        Doctor approved formula, 100% natural ingredients.
        Buy cheap meds with NO PRESCRIPTION needed!
        Guaranteed profit for our affiliate partners. Free money awaits!"""
    ),
    (
        "Financial Scam",
        """Make money fast with our risk free investment system!
        Double your income in just 30 days. No investment required.
        We guarantee financial freedom. Wire transfer payment available.
        Earn $5000 per week from home! This is URGENT."""
    ),
    (
        "Leet-Speak Spam (Obfuscated)",
        """Congr4tul4tions! You h4ve w0n fr33 m0ney!
        Cl4im your c4sh pr1ze now. 4ct n0w before 1t exP1res!
        S3lected w1nner n0tific4tion. L0ttery w1nner alert!!!"""
    ),
    (
        "Legitimate Newsletter (Should be HAM)",
        """Dear John, thank you for subscribing to our monthly digest.
        This month we cover natural language processing advances and
        new machine learning research. You can unsubscribe at any time.
        Privacy policy: example.com/privacy | Terms of service apply."""
    ),
    (
        "Legitimate Business Email (Should be HAM)",
        """Hi Sarah, I wanted to follow up on our meeting yesterday.
        The project deadline is urgent and we need to act now to ensure
        we deliver on time. Please confirm your availability for a call.
        Best regards, Tom — no spam, this is a direct business communication."""
    ),
    (
        "Borderline / Suspicious Email",
        """Hello, we have an urgent limited time offer for you.
        Our financial freedom course is now available at a discount.
        This is your last chance to join 10,000 successful students.
        Call us at 555-123-4567 to learn more."""
    ),
]

print(f"✅ Test dataset: {len(test_emails)} emails loaded.")

---
## 🚀 Part 6 — Run the Spam Detector

In [ ]:
results = []
print("\n" + "═"*68)
print("   AUTOMATA-BASED SPAM FILTER — ANALYSIS REPORT")
print("═"*68 + "\n")

for label, text in test_emails:
    result = fsa.classify(text)
    results.append((label, result))
    print_report(text, result, label=label)

---
## 📊 Part 7 — Summary Statistics & Evaluation

In [ ]:
print("\n" + "═"*60)
print("   SUMMARY: CLASSIFICATION RESULTS")
print("═"*60)
print(f"  {'#':<3} {'Email':<35} {'Score':<7} {'Verdict'}")
print("  " + "─"*58)

spam_count      = 0
suspicious_count = 0
ham_count       = 0

for i, (label, result) in enumerate(results, 1):
    short = label[:33]
    state_icon = {'SPAM': '🚫', 'SUSPICIOUS': '⚠️ ', 'CLEAN': '✅'}.get(result.final_state, '?')
    print(f"  {i:<3} {short:<35} {result.score:<7} {state_icon} {result.final_state}")
    if result.final_state == 'SPAM':       spam_count += 1
    elif result.final_state == 'SUSPICIOUS': suspicious_count += 1
    else:                                   ham_count += 1

total = len(results)
print("  " + "─"*58)
print(f"  Total emails    : {total}")
print(f"  🚫 SPAM         : {spam_count}  ({spam_count/total*100:.0f}%)")
print(f"  ⚠️  SUSPICIOUS   : {suspicious_count}  ({suspicious_count/total*100:.0f}%)")
print(f"  ✅ HAM (Clean)  : {ham_count}  ({ham_count/total*100:.0f}%)")
print("═"*60)

---
## 🔬 Part 8 — Pattern Complexity Analysis

In [ ]:
print("\n" + "═"*65)
print("   PATTERN COMPLEXITY ANALYSIS")
print("═"*65)

print("\n[1] Keyword Count by Category:")
for cat, kws in SPAM_KEYWORDS.items():
    avg_weight = sum(w for _, w in kws) / len(kws)
    print(f"   {cat:<20} {len(kws):>3} keywords   avg weight: {avg_weight:.2f}")

print("\n[2] Weight Distribution:")
all_kws = [(kw, w) for kws in SPAM_KEYWORDS.values() for kw, w in kws]
for weight in [1, 2, 3]:
    count = sum(1 for _, w in all_kws if w == weight)
    label = {1: 'Low', 2: 'Medium', 3: 'High'}[weight]
    bar = '▓' * count
    print(f"   Weight {weight} ({label:<6}): {bar} ({count})")

print("\n[3] Regex Pattern Sample (first 5 patterns):")
for compiled, cat, weight, kw in engine.compiled_patterns[:5]:
    print(f"   Keyword  : '{kw}'")
    print(f"   Pattern  : {compiled.pattern[:80]}")
    print(f"   Category : {cat}, Weight: {weight}")
    print()

print("[4] Leet-Speak Normalization Demo:")
leet_tests = ["fr33 m0ney", "4ct n0w", "w1nn3r", "c4sh pr1ze"]
for lt in leet_tests:
    normalized = engine._normalize_leet(lt)
    print(f"   '{lt}' → '{normalized}'")

print("\n[5] False Positive Reduction Signals:")
for i, (pattern, delta) in enumerate(SpamFSA.FP_WHITELIST_PATTERNS, 1):
    print(f"   {i}. Pattern: {pattern:<45} Adjustment: {delta:+d}")
print("─"*65)

---
## 🎯 Part 9 — Interactive: Test Your Own Email

In [ ]:
# ── Change the text below to test any email ──────────────────
custom_email = """
You have been selected as our lucky draw winner!
Claim your cash prize of $10,000 — act now, limited time offer!
Click here to verify your account details.
"""
# ─────────────────────────────────────────────────────────────

custom_result = fsa.classify(custom_email)
print_report(custom_email, custom_result, label="Custom Email Test")

---
## 📘 Part 10 — Theory: FSA Design Summary

```
FSA STATE DIAGRAM
═════════════════

             low                   low
 [START] ──────────► [CLEAN] ◄────────────┐
    │                   │                  │
    │ medium            │ medium           │
    ▼                   ▼                  │
 [SUSPICIOUS] ◄────────────────────────────┤
    │                   │ high             │
    │ high              ▼                  │
    └──────────────► [SPAM] ───────────────┘
                     (trap state)

Signals:
  low    = score < 3
  medium = 3 ≤ score < 7
  high   = score ≥ 7

Score = Σ(keyword weights) + amplifiers − false_positive_reducers
```

### Key Design Decisions

| Feature | Implementation | Benefit |
|---|---|---|
| Keyword Matching | Regex with `\\b` anchors | Exact word boundary, avoids partial matches |
| Pattern Complexity | Flexible `.` and `-` between chars | Catches obfuscated spam |
| Leet Normalization | Pre-processing substitution table | Catches `fr33`, `m0ney` etc. |
| False Positive Reduction | Whitelist context patterns | Reduces misclassification of legit emails |
| Amplifiers | Caps, `!!!`, suspicious URLs | Boosts score for obvious spam signals |
| Weighted Keywords | Weight 1–3 by domain | Prioritizes high-confidence signals |